# Atividade · Regressão linear múltipla na prática
**Fundamentos de Machine Learning · Aula 2 · Tempo sugerido: 1 hora**

Agora é sua vez. Aplique a receita do notebook de passo a passo a uma base nova.

**Como resolver**
- Complete cada `___` marcado com `# COMPLETE`.
- Execute as células de **verificação**. Se aparecer ✔, siga em frente.
- Responda as **questões abertas** na célula de texto logo abaixo de cada uma.

| Parte | Conteúdo | Tempo |
|---|---|---|
| 1 | Carregar o CSV | 5 min |
| 2 | Explorar e limpar a base | 15 min |
| 3 | X, y e treino/teste | 5 min |
| 4 | Treinar e interpretar | 10 min |
| 5 | Prever | 10 min |
| 6 | Visualizar o modelo | 10 min |
| 7 | Desafio | 5 min |

## A base de dados: *Auto MPG*

A base vem do **UCI Machine Learning Repository** (dataset id 9, licença CC BY 4.0) e também está no Kaggle (*uciml/autompg-dataset*). Cada linha é um **modelo de carro** dos anos 1970 e início dos 1980.

| Coluna | Significado |
|---|---|
| `mpg` | consumo em milhas por galão (**alvo**; maior = mais econômico) |
| `cylinders` | número de cilindros |
| `displacement` | cilindrada do motor (polegadas cúbicas) |
| `horsepower` | potência (hp). **Tem valores ausentes** |
| `weight` | peso (libras) |
| `acceleration` | tempo de 0 a 60 mph (segundos) |
| `model_year` | ano do modelo (70 = 1970) |
| `origin` | região de origem: 1 = EUA, 2 = Europa, 3 = Japão |
| `car_name` | nome do carro (identificador, não é feature) |

**Pergunta:** a partir das características de um carro, qual é o consumo esperado?

**Como obter o arquivo:** baixe `auto-mpg.csv` do Kaggle e salve-o em `data/`. Se preferir a fonte oficial da UCI, rode a célula opcional logo abaixo: ela usa o pacote `ucimlrepo` e grava o mesmo CSV.

In [ ]:
# OPCIONAL: baixar da UCI e salvar como CSV (requer: pip install ucimlrepo)
# from ucimlrepo import fetch_ucirepo
# auto = fetch_ucirepo(id=9)
# auto.data.original.to_csv('auto-mpg.csv', index=False)

## Parte 0 · Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

## Parte 1 · Carregar o CSV

**Tarefa 1.** Leia o arquivo com `pd.read_csv`. No arquivo do Kaggle, os valores ausentes de `horsepower` aparecem como `'?'`; o parâmetro `na_values='?'` os converte em `NaN`.

A segunda linha padroniza os nomes das colunas (`model year` vira `model_year`), para que o código funcione com o CSV do Kaggle e com o da UCI.

In [ ]:
df = pd.read_csv(___, na_values='?')                     # COMPLETE: caminho do arquivo
df.columns = df.columns.str.strip().str.replace(' ', '_')
df.___()                                                 # COMPLETE: 5 primeiras linhas

In [ ]:
# Verificação
assert {'mpg', 'horsepower', 'weight', 'model_year'} <= set(df.columns)
assert pd.api.types.is_numeric_dtype(df['horsepower']), 'horsepower deveria ser numérica: confira o na_values'
print('✔ Tarefa 1 ok')

## Parte 2 · Explorar e limpar a base

**Tarefa 2.** Mostre o tamanho da base, os tipos das colunas e a quantidade de valores ausentes por coluna.

In [ ]:
print(df.___)          # COMPLETE: (linhas, colunas)
df.___()               # COMPLETE: tipos e não nulos

In [ ]:
df.___().sum()         # COMPLETE: valores ausentes por coluna

**Tarefa 3.** Remova as linhas com valores ausentes e a coluna `car_name`, que é só um identificador. Guarde o resultado em `df_limpo`.

In [ ]:
df_limpo = df.___().drop(columns=___)        # COMPLETE
print(df_limpo.shape)

In [ ]:
# Verificação
assert df_limpo.isna().sum().sum() == 0 and 'car_name' not in df_limpo
print(f'✔ Tarefa 3 ok: {len(df) - len(df_limpo)} linhas removidas')

**Tarefa 4.** Mostre as estatísticas descritivas e a correlação de cada coluna com `mpg`, ordenada do menor para o maior valor.

In [ ]:
display(df_limpo.describe().round(1))
df_limpo.corr()[___].sort_values()           # COMPLETE: coluna alvo

**Tarefa 5.** Faça um gráfico de dispersão de cada feature contra `mpg` (use a lista `features`, já definida).

In [ ]:
features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for ax, col in zip(axes.ravel(), features):
    ax.scatter(___, ___, alpha=0.5)          # COMPLETE: feature no x, mpg no y
    ax.set_xlabel(col)
for ax in axes[:, 0]:
    ax.set_ylabel('mpg')
plt.tight_layout()
plt.show()

> **Questão aberta 1.** Quais features têm relação mais forte com `mpg`? A relação é positiva ou negativa? Em qual gráfico a relação parece **curva**, e não reta?

*Sua resposta:*

> **Questão aberta 2.** A coluna `origin` usa os números 1, 2 e 3 para EUA, Europa e Japão. Por que não faz sentido tratá-la como uma quantidade (por exemplo, achar que Japão = 3 × EUA)? Por isso ela ficou de fora da lista `features`; o desafio da Parte 7 mostra como incluí-la.

*Sua resposta:*

## Parte 3 · X, y e treino/teste

**Tarefa 6.** Monte `X` com as colunas da lista `features` e `y` com `mpg`. Separe 20% para teste com `random_state=42`.

In [ ]:
X = df_limpo[___]                  # COMPLETE
y = df_limpo[___]                  # COMPLETE

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=___, random_state=42)  # COMPLETE
print('Treino:', X_train.shape, '| Teste:', X_test.shape)

In [ ]:
# Verificação
assert X.shape[1] == 6 and X.ndim == 2
assert abs(len(X_test) / len(X) - 0.2) < 0.01
print('✔ Tarefa 6 ok')

## Parte 4 · Treinar e interpretar

**Tarefa 7.** Crie e treine o `LinearRegression` **apenas com o treino**. Monte uma tabela com o coeficiente de cada feature.

In [ ]:
modelo = ___                         # COMPLETE: crie o modelo
modelo.___(___, ___)                 # COMPLETE: treine com o conjunto de treino

print(f'a (intercepto) = {modelo.intercept_:.3f}')
pd.DataFrame({'feature': features, 'b': modelo.___}).round(4)   # COMPLETE

In [ ]:
# Verificação
assert len(modelo.coef_) == 6
print('✔ Tarefa 7 ok')

> **Questão aberta 3.** Interprete o coeficiente de `weight` e o de `model_year` em uma frase cada, usando as unidades. Lembre: cada `b` vale **mantendo as outras features fixas**.
>
> **Questão aberta 4.** O coeficiente de `weight` parece muito pequeno perto do de `model_year`. Isso quer dizer que o peso importa pouco? (Dica: compare as escalas das duas colunas no `describe`.)

*Sua resposta:*

## Parte 5 · Prever

**Tarefa 8.** Faça a previsão para o conjunto de teste e monte uma tabela com o `mpg` real, o previsto e o resíduo das 8 primeiras linhas.

In [ ]:
y_pred = modelo.___(___)                         # COMPLETE

comparacao = pd.DataFrame({
    'mpg_real': y_test,
    'mpg_previsto': y_pred.round(1),
    'residuo': ___,                              # COMPLETE: real - previsto
})
comparacao.head(8)

**Tarefa 9.** Preveja o consumo destes dois carros hipotéticos:

| | cylinders | displacement | horsepower | weight | acceleration | model_year |
|---|---|---|---|---|---|---|
| Compacto 1980 | 4 | 100 | 75 | 2200 | 16 | 80 |
| Sedã grande 1972 | 8 | 350 | 165 | 4200 | 11 | 72 |

In [ ]:
carros = pd.DataFrame({
    'cylinders':    [4, 8],
    'displacement': [100, 350],
    'horsepower':   [75, 165],
    'weight':       [2200, 4200],
    'acceleration': [16, 11],
    'model_year':   [80, 72],
}, index=['Compacto 1980', 'Sedã grande 1972'])

carros['mpg_previsto'] = ___                     # COMPLETE
carros

In [ ]:
# Verificação
assert carros.loc['Compacto 1980', 'mpg_previsto'] > carros.loc['Sedã grande 1972', 'mpg_previsto']
print('✔ Tarefa 9 ok')

## Parte 6 · Visualizar o modelo

**Tarefa 10.** Desenhe a **reta otimizada de `weight`**: varie só o peso e fixe as outras features na média do treino, como na Etapa 8.1 do passo a passo.

In [ ]:
medias = X_train.mean()
grade = pd.DataFrame([medias] * 100)
grade['weight'] = np.linspace(X['weight'].min(), X['weight'].max(), 100)

plt.figure(figsize=(8, 5))
plt.scatter(X_train['weight'], y_train, alpha=0.35, label='treino')
plt.scatter(X_test['weight'], y_test, alpha=0.8, marker='x', label='teste')
plt.plot(grade['weight'], ___, color='red', linewidth=2.5, label='reta otimizada')   # COMPLETE
plt.xlabel('weight (libras)'); plt.ylabel('mpg'); plt.legend()
plt.show()

**Tarefa 11.** Faça o gráfico **real vs. previsto** do conjunto de teste, com a diagonal `previsto = real`.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(___, ___, alpha=0.8)                 # COMPLETE: real no x, previsto no y
lim = [y.min() - 2, y.max() + 2]
plt.plot(lim, lim, 'k--', label='previsto = real')
plt.xlim(lim); plt.ylim(lim)
plt.xlabel('mpg real'); plt.ylabel('mpg previsto'); plt.legend()
plt.show()

> **Questão aberta 5.** Olhe o gráfico real vs. previsto. Para quais carros (consumo baixo, médio ou alto) os pontos se afastam mais da diagonal? Relacione com a curvatura que você viu na Questão 1.

*Sua resposta:*

## Parte 7 · Desafio

**Tarefa 12 (desafio).** Inclua `origin` no modelo do jeito certo. `pd.get_dummies` transforma a coluna em colunas indicadoras (0 ou 1), uma por região. Com `drop_first=True`, a primeira região (EUA) vira a referência.

Treine um novo modelo com as features anteriores **mais** as colunas indicadoras e compare o SSE no teste com o do modelo da Parte 4.

In [ ]:
df_d = df_limpo.copy()
df_d['origin'] = df_d['origin'].map({1: 'EUA', 2: 'Europa', 3: 'Japao'})
df_d = pd.get_dummies(df_d, columns=['origin'], drop_first=True, dtype=int)

features_d = features + ['origin_Europa', 'origin_Japao']
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    df_d[features_d], df_d['mpg'], test_size=0.2, random_state=42)

# Seu código aqui: treine modelo_d, calcule o SSE no teste dos dois modelos e mostre os coeficientes de origin

> **Questão aberta 6.** Como se interpreta o coeficiente de `origin_Japao`? Por que precisamos deixar uma região como referência (`drop_first=True`)?

*Sua resposta:*